# PyTorch ve VGG16 ile Fındık Görüntüleri Sınıflandırma Projesi

Bu proje, derin öğrenme teknikleri kullanılarak fındık görüntülerinin kalitesine göre sınıflandırılmasını hedeflemektedir. Python 3.14 uyumluluğu için **PyTorch** kütüphanesi ve önceden eğitilmiş **VGG16** mimarisi kullanılmıştır.

## Proje Bilgileri:
- **Veri Seti:** `hazel` veri seti (crack, cut, good, hole, print).
- **Model:** VGG16 (Transfer Learning).
- **Teknikler:** Data Augmentation, Dropout, CrossEntropyLoss.
- **Eğitim/Test:** %80 Eğitim, %20 Test.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from sklearn.model_selection import train_test_split

## 1. Veri Hazırlama ve Veri Çoğaltma

Görüntüler okunurken eğitim için veri çoğaltma (augmentation) uygulanmaktadır.

In [ ]:
data_dir = r'C:/Users/user/Desktop/hazel/test' # Test klasörünü ana veri seti kökü olarak kullanalım (veya tüm klasörleri birleştirelim)
# Not: Kullanıcının klasör yapısına göre tüm alt klasörlerin olduğu ana dizini seçiyoruz.
base_path = r'C:/Users/user/Desktop/hazel'

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Tüm veriyi yükle
full_dataset = datasets.ImageFolder(data_dir)
train_idx, test_idx = train_test_split(list(range(len(full_dataset))), test_size=0.2, random_state=42)

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

train_subset = Subset(full_dataset, train_idx)
test_subset = Subset(full_dataset, test_idx)

train_data = MyDataset(train_subset, transform=transform_train)
test_data = MyDataset(test_subset, transform=transform_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = full_dataset.classes
print(f"Sınıflar: {class_names}")

## 2. VGG16 Model Mimarisinin Kurulması

Önceden eğitilmiş VGG16 modelini yüklüyoruz ve son katmanlarını 5 sınıflı fındık verimize göre güncelliyoruz.

In [ ]:
model = models.vgg16(weights='IMAGENET1K_V1')

# Parametreleri dondurma
for param in model.features.parameters():
    param.requires_grad = False

# Sınıflandırıcı kısmını değiştirme
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 5) # 5 sınıf (crack, cut, good, hole, print)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

## 3. Model Eğitimi (10 Epoch)

Eğitim ve validasyon süreçlerini başlatıyoruz.

In [ ]:
epochs = 10
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)
    
    # Validasyon
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_losses.append(val_loss / len(test_loader))
    val_accs.append(val_correct / val_total)
    
    print(f"Epoch {epoch+1}/{epochs} - Kayıp: {train_losses[-1]:.4f} - Başarım: {train_accs[-1]:.4f} - Val Kayıp: {val_losses[-1]:.4f} - Val Başarım: {val_accs[-1]:.4f}")

## 4. Kayıp ve Başarım Grafikleri

Eğitim sürecindeki gelişimi görselleştirelim.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_accs, label='Eğitim Başarımı')
plt.plot(val_accs, label='Test Başarımı')
plt.title('Doğruluk Grafik')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Eğitim Kaybı')
plt.plot(val_losses, label='Test Kaybı')
plt.title('Kayıp Grafik')
plt.legend()
plt.show()

## 5. Değerlendirme ve Metrikler

Accuracy, Precision, Recall ve F1 Score değerlerini hesaplıyoruz.

In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Sınıflandırma Raporu:")
print(classification_report(y_true, y_pred, target_names=class_names))

print(f"Doğruluk (Accuracy): {accuracy_score(y_true, y_pred):.4f}")
print(f"Keskinlik (Precision): {precision_score(y_true, y_pred, average='weighted'):.4f}")
print(f"Duyarlılık (Recall): {recall_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 Skoru: {f1_score(y_true, y_pred, average='weighted'):.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Tahmini Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Karmaşıklık Matrisi')
plt.show()